# Qwen3.8-27B-Uncensored (FastMTP) — A100 Autonomous Agent & REA Server

This notebook provisions and hosts **`HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF`** on an **NVIDIA A100-SXM4 (40GB/80GB)**.

### Key Features:
- **Zero Refusals (0/465)**: Uncensored aggressive profile for deep reverse engineering, deobfuscation, and unconstrained agentic coding.
- **HauhauCS FastMTP**: Up to 3x token generation speed using speculative decoding sidecar.
- **Massive Context**: 65,536 tokens enabled with FlashAttention-2.
- **OpenAI-Compatible Server**: Exposed securely via Cloudflare Tunnel (`trycloudflare.com`) on port 8088.
- **Client Integration**: Seamlessly connect your laptop's **Antigravity, Aider, Continue.dev, LM Studio, or Python scripts**.

In [ ]:
# ============================================================
# Cell 1: Hardware & CUDA Verification
# ============================================================
import torch, os
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({total_vram:.2f} GB VRAM)")
    if total_vram < 20:
        print("[WARNING] GPU has less than 20GB VRAM. Consider switching to A100 or L4!")
else:
    print("[ERROR] No GPU detected! Go to Runtime -> Change runtime type -> Select A100/L4 GPU.")

In [ ]:
# ============================================================
# Cell 2: Install Cloudflared & Build llama.cpp with FastMTP
# ============================================================
!apt-get update -qq && apt-get install -y -qq cmake build-essential curl aria2
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1
!pip install -q huggingface_hub

import os
if not os.path.exists("llama.cpp"):
    print("Cloning llama.cpp & applying FastMTP patch...")
    !git clone https://github.com/ggerganov/llama.cpp
    %cd llama.cpp
    !git checkout 4df29be4f4c3673f428170fda944a5b19f743bb8
    !curl -sL -o HauhauCS-FastMTP-llama.cpp.patch https://huggingface.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF/resolve/main/HauhauCS-FastMTP-llama.cpp.patch
    !git apply HauhauCS-FastMTP-llama.cpp.patch
    !cmake -S . -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
    !cmake --build build --config Release -j$(nproc)
    %cd ..
    print("[SUCCESS] llama.cpp built with FastMTP and CUDA!")
else:
    print("[INFO] llama.cpp directory already exists.")

In [ ]:
# ============================================================
# Cell 3: Download Model Weights & FastMTP Sidecar
# ============================================================
from huggingface_hub import hf_hub_download
import os

REPO_ID = "HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF"
# On A100 (40GB/80GB), Q5_K_P (20.22 GB) or Q8_K_P (31.46 GB) delivers maximum fidelity
MODEL_FILE = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q5_K_P.gguf"
DRAFT_FILE = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-FastMTP-32K.gguf"

os.makedirs("./models", exist_ok=True)

print(f"Downloading {MODEL_FILE}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=MODEL_FILE, local_dir="./models")

print(f"Downloading FastMTP Draft sidecar {DRAFT_FILE}...")
draft_path = hf_hub_download(repo_id=REPO_ID, filename=DRAFT_FILE, local_dir="./models")

print("\n[READY] Weights downloaded successfully to ./models/")

In [ ]:
# ============================================================
# Cell 4: Launch Cloudflare Tunnel & llama-server on Port 8088
# ============================================================
import subprocess, time, re, sys

# 0. Clean up any previous background processes
subprocess.run(["pkill", "-9", "-f", "llama-server"])
subprocess.run(["pkill", "-9", "-f", "cloudflared"])
time.sleep(1)

PORT = 8088
MODEL_PATH = f"./models/{MODEL_FILE}"
DRAFT_PATH = f"./models/{DRAFT_FILE}"

# 1. Start Cloudflare Tunnel in background on port 8088
print(f"[*] Starting Cloudflare tunnel on port {PORT}...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Failed to acquire Cloudflare Tunnel URL within 30s.")

print("=" * 70)
print(f"[CONNECTED] REMOTE API BASE URL: {public_url}/v1")
print(f"Copy this URL to your laptop config (OPENAI_BASE_URL={public_url}/v1)")
print("=" * 70)

# 2. Build verified llama-server command on 127.0.0.1:8088
server_cmd = [
    "./llama.cpp/build/bin/llama-server",
    "-m", MODEL_PATH,
    "--spec-draft-model", DRAFT_PATH,
    "--spec-draft-ngl", "all",
    "--spec-type", "draft-mtp",
    "--spec-draft-n-max", "3",
    "--spec-draft-p-min", "0",
    "--ctx-size", "65536",
    "--parallel", "1",
    "-ngl", "999",
    "--batch-size", "2048",
    "--ubatch-size", "512",
    "-fa", "on",
    "--jinja",
    "--host", "127.0.0.1",
    "--port", str(PORT)
]

print(f"[*] Launching llama-server on port {PORT}:\n{' '.join(server_cmd)}\n")
server_proc = subprocess.Popen(
    server_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in server_proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n[*] Stopping server and tunnel...")
    server_proc.terminate()
    tunnel_proc.terminate()